# Nifty50 Experiment Runner

Workmanlike Colab runner for long multimodal experiments. It installs the project, mounts Google Drive when available, builds a fresh real-world multimodal artifact, checks modality independence before training, runs all ablation variants with purged walk-forward CV, runs the corrected backtest on the best variant, and writes a consolidated `summary.md` under the output directory.

Expected runtime depends on universe size, history length, GPU availability, and FinBERT download/cache state. The default 6-stock smoke configuration is intended to finish within a Colab session; 50 stocks over 5 years is a long unattended run. A separate demo notebook will explain the results later; this notebook only produces experiment artifacts.

In [ ]:
%%capture setup_log
# Setup - environment. Re-run is safe; pip output is captured unless this cell fails.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Randhir123/nifty50-multimodal-transformer.git"
REPO_DIR = Path("/content/nifty50-multimodal-transformer")

if not Path("pyproject.toml").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
print(f"Repository ready at {Path.cwd()}")

In [ ]:
# Setup - Google Drive mount. Outside Colab this falls back to local output paths.
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted at /content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive mount skipped: {exc}")
    print("Outputs will use the configured path if available, otherwise a local fallback.")

In [ ]:
# =====================================================================
# === EXPERIMENT CONFIG - EDIT THIS CELL ONLY ==========================
# =====================================================================

TICKERS = "RELIANCE.NS,TCS.NS,INFY.NS,HDFCBANK.NS,ICICIBANK.NS,SBIN.NS"
# Comma-separated NSE tickers. Default is the 6-stock smoke set.
# For full Nifty 50 expansion, paste the full list here.

PERIOD = "1y"
# yfinance period string. Use "1y", "2y", "5y", "10y", or "max".
# Longer periods need more compute and exercise more market regimes.

WINDOW_SIZE = 20
HORIZON_DAYS = 3
# OHLCV window length and forward-return horizon. These match the leakage-test assumptions.

EPOCHS = 20
BATCH_SIZE = 16
# Training schedule per ablation variant per fold.

CV_SPLITS = 3
EMBARGO_DAYS = 3
# Walk-forward purged CV. Embargo is in calendar days.

OUTPUT_ROOT = "/content/drive/MyDrive/nifty50_experiments"
# Where artifacts are written. A timestamped subdirectory is created.
# Outside Colab, this falls back to ./outputs/nifty50_experiments if Drive is unavailable.

RAW_DATA_DIR = ""
# Optional: paste a previous run's raw folder to reuse downloaded OHLCV CSVs.
# Example: "/content/drive/MyDrive/nifty50_experiments/20260510_123456/raw"
# Leave blank to use this run's OUTPUT_DIR/raw and download missing files only.

FORCE_REFRESH = False
# Keep False to use cached ticker CSVs when present. Set True only when you intentionally want fresh yfinance downloads.

DEVICE = "cuda"
# Set to "cpu" if no GPU is available. The next cell will fall back automatically if needed.

In [ ]:
# Resolved paths and fixed experiment constants.
import json
import os
import random
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch

RANDOM_SEED = 42  # Fixed by design. Multi-seed evaluation needs a different runner structure.
MODEL_DIM = 16    # Session-5 trainer-collapse-safe compact model config.
NUM_HEADS = 4
NUM_LAYERS = 1
FF_DIM = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
POOLING = "mean"
BENCHMARK = "^NSEI"
ABLATION_VARIANTS = ["tabular_only", "tabular_kg", "tabular_image", "tabular_text", "tabular_image_text_kg"]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
else:
    if DEVICE == "cuda":
        print("CUDA requested but unavailable; falling back to CPU.")
    DEVICE = "cpu"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
if OUTPUT_ROOT.startswith("/content/drive") and not Path("/content/drive").exists():
    OUTPUT_ROOT = "outputs/nifty50_experiments"
OUTPUT_DIR = Path(OUTPUT_ROOT) / RUN_ID
RAW_DIR = Path(RAW_DATA_DIR).expanduser() if RAW_DATA_DIR.strip() else OUTPUT_DIR / "raw"
ABLATION_DIR = OUTPUT_DIR / "ablations"
BACKTEST_DIR = OUTPUT_DIR / "backtest"
for path in [OUTPUT_DIR, RAW_DIR, ABLATION_DIR, BACKTEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TICKER_LIST = [ticker.strip() for ticker in TICKERS.split(",") if ticker.strip()]
END_DATE = date.today().isoformat()

def resolve_start_date(period: str, end: str) -> str:
    if period == "max":
        return "1990-01-01"
    end_date = pd.Timestamp(end).date()
    if period.endswith("y"):
        return (end_date - timedelta(days=int(period[:-1]) * 365)).isoformat()
    if period.endswith("mo"):
        return (end_date - timedelta(days=int(period[:-2]) * 31)).isoformat()
    raise ValueError("PERIOD must look like '1y', '5y', '10y', '9mo', or 'max'.")

START_DATE = resolve_start_date(PERIOD, END_DATE)
EXPERIMENT_START_TIME = time.time()
STEP_RESULTS = {"config": {}}
STEP_ERRORS = {}

CONFIG = {
    "tickers": TICKER_LIST,
    "period": PERIOD,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "window_size": WINDOW_SIZE,
    "horizon_days": HORIZON_DAYS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "cv_splits": CV_SPLITS,
    "embargo_days": EMBARGO_DAYS,
    "output_dir": str(OUTPUT_DIR),
    "raw_dir": str(RAW_DIR),
    "force_refresh": FORCE_REFRESH,
    "device": DEVICE,
    "seed": RANDOM_SEED,
    "model_dim": MODEL_DIM,
    "variants": ABLATION_VARIANTS,
}
STEP_RESULTS["config"] = CONFIG
print(json.dumps(CONFIG, indent=2))

In [ ]:
# Build multimodal samples with retry around yfinance and FinBERT download/encoding.
import dataclasses
import gc
import traceback
from argparse import Namespace
from pathlib import Path

from scripts.run_real_world_demo import (
    DEFAULT_SECTORS,
    FEATURE_COLUMNS,
    _attach_finbert_text_tokens,
    _build_event_records,
    _build_kg_returns,
    _build_tabular_rows,
    _build_text_records,
)
from src.data.download_yfinance import (
    deterministic_csv_path_for_ticker,
    download_benchmark_data,
    download_single_ticker_ohlcv,
    save_ticker_csv,
)
from src.data.multimodal_samples import (
    attach_gaf_mtf_image_tokens,
    attach_kg_tokens,
    build_tabular_multimodal_samples,
    save_multimodal_samples,
)
from src.kg.build_graph import build_market_knowledge_graph


def retry_call(label, fn, attempts=3, base_sleep=5):
    last_exc = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            print(f"{label} failed on attempt {attempt}/{attempts}: {exc}")
            if attempt < attempts:
                time.sleep(base_sleep * (2 ** (attempt - 1)))
    raise last_exc


def load_or_download_with_retries(tickers, benchmark, start, end, raw_dir, force_refresh=False):
    provenance = {}
    stock_data = {}
    today = date.today()
    failures = {}

    for ticker in tickers:
        csv_path = deterministic_csv_path_for_ticker(ticker, raw_dir)
        if csv_path.exists() and not force_refresh:
            stock_data[ticker] = pd.read_csv(csv_path, parse_dates=["date"])
            provenance[ticker] = f"cache:{csv_path}"
            continue
        try:
            df = retry_call(
                f"download {ticker}",
                lambda ticker=ticker: download_single_ticker_ohlcv(ticker, start=start, end=end),
            )
            save_path = save_ticker_csv(ticker, df, output_dir=raw_dir)
            stock_data[ticker] = df
            provenance[ticker] = f"download:{save_path}"
        except Exception as exc:
            failures[ticker] = repr(exc)
            print(f"Skipping {ticker} after retries: {exc}")

    if len(stock_data) < max(1, len(tickers) // 2):
        raise RuntimeError(f"Only {len(stock_data)}/{len(tickers)} tickers succeeded; failures={failures}")

    benchmark_path = deterministic_csv_path_for_ticker(benchmark, raw_dir)
    if benchmark_path.exists() and not force_refresh:
        benchmark_df = pd.read_csv(benchmark_path, parse_dates=["date"])
        provenance[benchmark] = f"cache:{benchmark_path}"
    else:
        benchmark_df = retry_call(
            f"download {benchmark}",
            lambda: download_benchmark_data(benchmark, start=start, end=end),
        )
        save_path = save_ticker_csv(benchmark, benchmark_df, output_dir=raw_dir)
        provenance[benchmark] = f"download:{save_path}"

    return stock_data, benchmark_df, provenance, failures

try:
    stock_data, benchmark_df, provenance, download_failures = load_or_download_with_retries(
        TICKER_LIST, BENCHMARK, START_DATE, END_DATE, RAW_DIR, force_refresh=FORCE_REFRESH
    )

    tabular_df = _build_tabular_rows(
        stock_data=stock_data,
        benchmark_df=benchmark_df,
        horizon_days=HORIZON_DAYS,
    )
    TABULAR_CSV = OUTPUT_DIR / "tabular_samples.csv"
    tabular_df.to_csv(TABULAR_CSV, index=False)

    text_records = _build_text_records(tabular_df)
    TEXT_RECORDS_CSV = OUTPUT_DIR / "text_records.csv"
    text_records.to_csv(TEXT_RECORDS_CSV, index=False)

    sectors = {ticker: DEFAULT_SECTORS.get(ticker, "Unknown") for ticker in stock_data.keys()}
    stock_sectors = pd.DataFrame([{"stock_id": ticker, "sector_id": sector} for ticker, sector in sectors.items()])
    stock_sectors.to_csv(OUTPUT_DIR / "stock_sectors.csv", index=False)

    kg_returns = _build_kg_returns(tabular_df)
    kg_returns.to_csv(OUTPUT_DIR / "kg_returns.csv", index=False)
    event_records = _build_event_records(tabular_df)
    event_records.to_csv(OUTPUT_DIR / "event_records.csv", index=False)

    arrays = build_tabular_multimodal_samples(tabular_df, feature_cols=FEATURE_COLUMNS, window_size=WINDOW_SIZE)
    arrays = retry_call("FinBERT text encoding", lambda: _attach_finbert_text_tokens(arrays, text_records, device=DEVICE), attempts=3, base_sleep=10)
    graph = build_market_knowledge_graph(sectors, event_records=event_records)
    arrays = attach_kg_tokens(arrays, graph, returns=kg_returns)
    arrays = attach_gaf_mtf_image_tokens(arrays, raw_dir=RAW_DIR, image_size=32, output_dim=MODEL_DIM, device=DEVICE)

    DATASET_PATH = save_multimodal_samples(arrays, OUTPUT_DIR / "real_world_multimodal_samples.npz")
    STEP_RESULTS["data_build"] = {
        "dataset_path": str(DATASET_PATH),
        "tabular_csv": str(TABULAR_CSV),
        "text_records_csv": str(TEXT_RECORDS_CSV),
        "download_failures": download_failures,
        "provenance": provenance,
    }
    del stock_data, benchmark_df, text_records, kg_returns, event_records, graph
    gc.collect()
except Exception as exc:
    STEP_ERRORS["data_build"] = traceback.format_exc()
    print(STEP_ERRORS["data_build"])

In [ ]:
# Artifact summary. Stop here manually if the sample count or label balance looks wrong.
try:
    data = np.load(DATASET_PATH, allow_pickle=False)
    y = data["y"]
    dates = pd.to_datetime(data["end_dates"])
    artifact_summary = {
        "samples": int(y.shape[0]),
        "date_min": str(dates.min().date()),
        "date_max": str(dates.max().date()),
        "positive_label_rate": float(np.mean(y)),
        "tabular_shape": list(data["tabular_tokens"].shape),
        "image_shape": list(data["image_tokens"].shape) if "image_tokens" in data.files else None,
        "text_shape": list(data["text_tokens"].shape) if "text_tokens" in data.files else None,
        "kg_shape": list(data["kg_tokens"].shape) if "kg_tokens" in data.files else None,
    }
    STEP_RESULTS["artifact_summary"] = artifact_summary
    print(json.dumps(artifact_summary, indent=2))
except Exception:
    STEP_ERRORS["artifact_summary"] = traceback.format_exc()
    print(STEP_ERRORS["artifact_summary"])

In [ ]:
# Modality independence check before training.
try:
    from scripts.check_modality_independence import _mean_abs_corr, _prepare

    raw = {}
    data = np.load(DATASET_PATH, allow_pickle=False)
    for key, label in [
        ("tabular_tokens", "tabular"),
        ("text_tokens", "text"),
        ("image_tokens", "image"),
        ("kg_tokens", "kg"),
    ]:
        if key in data.files:
            raw[label] = _prepare(data[key], max_dim=50)

    names = list(raw.keys())
    table = {
        i: {j: (1.0 if i == j else _mean_abs_corr(raw[i], raw[j])) for j in names}
        for i in names
    }
    independence_df = pd.DataFrame(table, index=names)
    INDEPENDENCE_CSV = OUTPUT_DIR / "modality_independence.csv"
    independence_df.to_csv(INDEPENDENCE_CSV)
    STEP_RESULTS["modality_independence"] = independence_df.round(4).to_dict()
    display(independence_df.style.format("{:.4f}"))
except Exception:
    STEP_ERRORS["modality_independence"] = traceback.format_exc()
    print(STEP_ERRORS["modality_independence"])

In [ ]:
# Run ablations with fold-level checkpoint resume.
try:
    import csv
    from scripts.run_ablation_study import (
        DEFAULT_VARIANTS,
        aggregate_fold_metrics,
        available_dataset_keys,
        load_checkpoint_metrics,
        load_checkpoint_predictions,
        select_variants,
        summarize_predictions,
        write_diagnostics,
        write_fold_results,
        write_results,
    )
    from src.training.cv import PurgedWalkForwardSplit
    from src.training.train_fusion import FusionArrays, load_fusion_arrays, slice_fusion_arrays, train_on_arrays

    def make_variant_arrays(full_arrays, variant):
        return FusionArrays(
            tabular_tokens=full_arrays.tabular_tokens,
            y=full_arrays.y,
            end_dates=full_arrays.end_dates,
            image_tokens=full_arrays.image_tokens if variant.use_image else None,
            text_tokens=full_arrays.text_tokens if variant.use_text else None,
            kg_tokens=full_arrays.kg_tokens if variant.use_kg else None,
            stock_ids=full_arrays.stock_ids,
        )

    train_args = Namespace(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        model_dim=MODEL_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        ff_dim=FF_DIM,
        val_fraction=0.25,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        dropout=DROPOUT,
        pooling=POOLING,
        max_tokens=4096,
    )

    dataset_keys = available_dataset_keys(DATASET_PATH)
    variants = select_variants(dataset_keys, variants=DEFAULT_VARIANTS, strict=True)
    full_arrays = load_fusion_arrays(
        DATASET_PATH,
        use_image="image_tokens" in dataset_keys,
        use_text="text_tokens" in dataset_keys,
        use_kg="kg_tokens" in dataset_keys,
    )
    splitter = PurgedWalkForwardSplit(n_splits=CV_SPLITS, horizon_days=HORIZON_DAYS, embargo_days=EMBARGO_DAYS)
    checkpoint_dir = ABLATION_DIR / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    results = []
    all_fold_rows = []
    for variant in variants:
        print(f"Running CV ablation variant: {variant.name}")
        variant_arrays = make_variant_arrays(full_arrays, variant)
        fold_metrics_list = []
        for cv_split in splitter.split(full_arrays.end_dates):
            fold_k = cv_split.fold
            checkpoint_path = checkpoint_dir / f"{variant.name}_fold{fold_k}.pt"
            if checkpoint_path.exists():
                print(f"  fold {fold_k}: using existing checkpoint {checkpoint_path}")
                fold_val_metrics = load_checkpoint_metrics(checkpoint_path)
            else:
                print(f"  fold {fold_k}: train={len(cv_split.train_idx)} val={len(cv_split.val_idx)}")
                train_arrays = slice_fusion_arrays(variant_arrays, cv_split.train_idx)
                val_arrays = slice_fusion_arrays(variant_arrays, cv_split.val_idx)
                fold_val_metrics = train_on_arrays(train_arrays, val_arrays, args=train_args, checkpoint_path=checkpoint_path)

            fold_metrics_list.append(fold_val_metrics)
            y_true, y_prob, end_dates, stock_ids = load_checkpoint_predictions(checkpoint_path)
            diagnostics = summarize_predictions(y_true, y_prob)
            all_fold_rows.append({
                "variant": variant.name,
                "fold": fold_k,
                "modalities": "+".join(variant.modalities),
                "checkpoint_path": str(checkpoint_path),
                **fold_val_metrics,
                **diagnostics,
            })
            write_fold_results(all_fold_rows, ABLATION_DIR)

        aggregated = aggregate_fold_metrics(fold_metrics_list)
        results.append({
            "variant": variant.name,
            "modalities": "+".join(variant.modalities),
            "n_folds": len(fold_metrics_list),
            "checkpoint_path": str(checkpoint_dir / f"{variant.name}_fold*.pt"),
            "prediction_scores_path": "",
            "command": f"notebook_cv:{len(fold_metrics_list)} folds",
            "accuracy": aggregated.get("accuracy_mean", ""),
            "precision": aggregated.get("precision_mean", ""),
            "recall": aggregated.get("recall_mean", ""),
            "f1": aggregated.get("f1_mean", ""),
            "roc_auc": aggregated.get("roc_auc_mean", ""),
            **aggregated,
        })
        write_results(results, ABLATION_DIR)
        write_diagnostics(results, ABLATION_DIR)

    ABLATION_RESULTS_CSV, ABLATION_RESULTS_JSON = write_results(results, ABLATION_DIR)
    write_diagnostics(results, ABLATION_DIR)
    ablation_df = pd.DataFrame(results).sort_values("roc_auc_mean", ascending=False)
    STEP_RESULTS["ablations"] = {
        "results_csv": str(ABLATION_RESULTS_CSV),
        "best_variant": str(ablation_df.iloc[0]["variant"]),
        "best_roc_auc_mean": float(ablation_df.iloc[0]["roc_auc_mean"]),
    }
    display(ablation_df[["variant", "roc_auc_mean", "roc_auc_std", "accuracy_mean", "f1_mean"]])
except Exception:
    STEP_ERRORS["ablations"] = traceback.format_exc()
    print(STEP_ERRORS["ablations"])

In [ ]:
# Aggregate CV prediction scores for the best variant and run the corrected backtest.
try:
    from scripts.run_ablation_study import load_checkpoint_predictions

    ablation_df = pd.read_csv(ABLATION_DIR / "ablation_results.csv")
    best_row = ablation_df.sort_values("roc_auc_mean", ascending=False).iloc[0]
    BEST_VARIANT = str(best_row["variant"])
    BEST_PREDICTIONS_CSV = ABLATION_DIR / f"prediction_scores_{BEST_VARIANT}_cv.csv"

    rows = []
    for fold_k in range(CV_SPLITS):
        ckpt = ABLATION_DIR / "checkpoints" / f"{BEST_VARIANT}_fold{fold_k}.pt"
        y_true, y_prob, end_dates, stock_ids = load_checkpoint_predictions(ckpt)
        y_pred = (y_prob >= 0.5).astype(int)
        for i, (label, prob, pred) in enumerate(zip(y_true, y_prob, y_pred)):
            rows.append({
                "row_id": f"fold{fold_k}_{i}",
                "fold": fold_k,
                "end_date": "" if end_dates is None else str(end_dates[i]),
                "stock_id": "" if stock_ids is None else str(stock_ids[i]),
                "y_true": int(label),
                "y_prob": float(prob),
                "y_pred": int(pred),
            })
    pd.DataFrame(rows).to_csv(BEST_PREDICTIONS_CSV, index=False)

    command = [
        sys.executable,
        "scripts/run_backtest.py",
        "--predictions", str(BEST_PREDICTIONS_CSV),
        "--tabular-samples", str(TABULAR_CSV),
        "--raw-dir", str(RAW_DIR),
        "--output-dir", str(BACKTEST_DIR),
        "--horizon-days", str(HORIZON_DAYS),
        "--top-k", "1",
    ]
    subprocess.run(command, check=True)

    with open(BACKTEST_DIR / "backtest_metrics.json", "r", encoding="utf-8") as handle:
        backtest_metrics = json.load(handle)
    STEP_RESULTS["backtest"] = {
        "best_variant": BEST_VARIANT,
        "predictions_csv": str(BEST_PREDICTIONS_CSV),
        "metrics": backtest_metrics,
        "plot": str(BACKTEST_DIR / "backtest_curve.png"),
    }
    print(json.dumps(STEP_RESULTS["backtest"], indent=2))
    try:
        from IPython.display import Image, display
        display(Image(filename=str(BACKTEST_DIR / "backtest_curve.png")))
    except Exception:
        pass
except Exception:
    STEP_ERRORS["backtest"] = traceback.format_exc()
    print(STEP_ERRORS["backtest"])

In [ ]:
# Write summary.md. This cell is intentionally defensive and should run even after partial failures.
def markdown_table(df, include_index=False):
    if df is None or df.empty:
        return "_No rows._"
    out = df.copy()
    if include_index:
        out = out.reset_index().rename(columns={"index": "modality"})
    headers = [str(c) for c in out.columns]
    rows = []
    for _, row in out.iterrows():
        rows.append([str(row[c]) for c in out.columns])
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines.extend("| " + " | ".join(row) + " |" for row in rows)
    return "\n".join(lines)

try:
    total_runtime_minutes = (time.time() - EXPERIMENT_START_TIME) / 60
except Exception:
    total_runtime_minutes = None

summary_lines = [
    "# Nifty50 Experiment Summary",
    "",
    f"- Run ID: `{globals().get('RUN_ID', 'unknown')}`",
    f"- Output directory: `{globals().get('OUTPUT_DIR', 'unknown')}`",
    f"- Total runtime minutes: `{total_runtime_minutes:.2f}`" if total_runtime_minutes is not None else "- Total runtime minutes: `unknown`",
    "",
    "## Config",
    "",
    "```json",
    json.dumps(globals().get("CONFIG", {}), indent=2, default=str),
    "```",
    "",
]

summary_lines += ["## Artifact", ""]
summary_lines += ["```json", json.dumps(STEP_RESULTS.get("artifact_summary", {}), indent=2, default=str), "```", ""]

summary_lines += ["## Modality Independence", ""]
try:
    summary_lines.append(markdown_table(pd.read_csv(OUTPUT_DIR / "modality_independence.csv", index_col=0), include_index=True))
except Exception as exc:
    summary_lines.append(f"_Not available: {exc}_")
summary_lines.append("")

summary_lines += ["## Ablation Results", ""]
try:
    cols = ["variant", "roc_auc_mean", "roc_auc_std", "accuracy_mean", "f1_mean"]
    df = pd.read_csv(ABLATION_DIR / "ablation_results.csv")
    summary_lines.append(markdown_table(df.loc[:, [c for c in cols if c in df.columns]]))
except Exception as exc:
    summary_lines.append(f"_Not available: {exc}_")
summary_lines.append("")

summary_lines += ["## Backtest", ""]
try:
    with open(BACKTEST_DIR / "backtest_metrics.json", "r", encoding="utf-8") as handle:
        metrics = json.load(handle)
    summary_lines += ["```json", json.dumps(metrics, indent=2), "```"]
except Exception as exc:
    summary_lines.append(f"_Not available: {exc}_")
summary_lines.append("")

summary_lines += ["## Errors", ""]
if STEP_ERRORS:
    for name, err in STEP_ERRORS.items():
        summary_lines += [f"### {name}", "", "```text", err[-4000:], "```", ""]
else:
    summary_lines.append("No recorded step errors.")
    summary_lines.append("")

SUMMARY_MD = OUTPUT_DIR / "summary.md"
SUMMARY_MD.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Summary written to {SUMMARY_MD}")

In [ ]:
# Done.
print("Done.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Summary: {SUMMARY_MD if 'SUMMARY_MD' in globals() else OUTPUT_DIR / 'summary.md'}")
if STEP_ERRORS:
    print("Completed with recorded errors; see summary.md for details.")